# Lab 28 — Few-shot adaptation với Logistic Regression + SMOTENC

## Mục tiêu

Colab này kiểm tra một câu hỏi cụ thể:

> Khi model LR + SMOTENC được huấn luyện trên **2 development hospitals**, việc có một support set rất nhỏ có nhãn từ hospital thứ 3 có giúp thích nghi với site đó hay không?

Đây là **few-shot target-site adaptation**, không phải few-shot learning theo nghĩa pre-trained foundation model. Để kiểm soát overfitting, model chỉ cập nhật **intercept** bằng support labels; toàn bộ feature coefficients được giữ nguyên.

## Protocol không leak

- Chỉ dùng Cleveland, Switzerland và VA trong thí nghiệm phát triển.
- Mỗi lượt chọn 1 hospital làm target-site, 2 hospital còn lại làm source train.
- Optuna và SMOTENC chỉ chạy trên source training data.
- Support set được lấy nhỏ, phân tầng theo class, từ target-site.
- Query set còn lại của target-site chỉ dùng để tính metric.
- Hungarian không được load, không được dùng để tuning, chọn support size, chọn seed hay chọn model.
- Kết quả ở đây là **development simulation**. External holdout Hungarian vẫn phải được giữ kín cho đánh giá cuối cùng; nếu đã xem kết quả Hungarian thì không còn blind tuyệt đối.

In [ ]:
!pip -q install optuna imbalanced-learn seaborn

In [ ]:
import json
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from IPython.display import display
from imblearn.over_sampling import SMOTENC
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 100)

FEATURES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]
TARGET = "target"
SITE = "site"
NUMERICAL_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = [
    "sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal",
]

BASE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease"
FILES = {
    "cleveland": "processed.cleveland.data",
    "switzerland": "processed.switzerland.data",
    "va": "processed.va.data",
}
COLUMNS = FEATURES + ["num"]
DEVELOPMENT_SITES = ["cleveland", "switzerland", "va"]

MODEL_SEEDS = (42, 123, 2025)
SUPPORT_SEEDS = (42, 123, 2025)
SUPPORT_PER_CLASS = (2, 5)
N_TRIALS = 10
THRESHOLD = 0.50
INTERCEPT_PRIOR_STRENGTH = 5.0

OUTPUT_DIR = Path("/content/uci_multicenter_fewshot_lr_smotenc_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_DIR_CANDIDATES = [
    Path("/content/heart-disease-diagnosis/data/raw/uci_multicenter"),
    Path("/content/data/raw/uci_multicenter"),
    Path("data/raw/uci_multicenter"),
    Path("../data/raw/uci_multicenter"),
]
LOCAL_DATA_DIR = next(
    (path for path in LOCAL_DATA_DIR_CANDIDATES if path.exists()),
    None,
)

print("Development sites:", DEVELOPMENT_SITES)
print("Models:", ["Logistic Regression + SMOTENC"])
print("Support per class:", SUPPORT_PER_CLASS)
print("Model seeds:", MODEL_SEEDS)
print("Support seeds:", SUPPORT_SEEDS)
print("External holdout loaded:", False)

In [ ]:
def read_uci(site, filename):
    source = (LOCAL_DATA_DIR / filename) if LOCAL_DATA_DIR else f"{BASE_URL}/{filename}"
    frame = pd.read_csv(
        source,
        names=COLUMNS,
        na_values=["?"],
        skipinitialspace=True,
    )
    frame = frame.apply(pd.to_numeric, errors="coerce")
    frame[TARGET] = (frame["num"] > 0).astype("int8")
    frame[SITE] = site
    return frame[FEATURES + [TARGET, SITE]]


development = pd.concat(
    [read_uci(site, filename) for site, filename in FILES.items()],
    ignore_index=True,
)

assert len(development) == 626
assert set(development[SITE]) == set(DEVELOPMENT_SITES)

site_summary = development.groupby(SITE).agg(
    rows=(TARGET, "size"),
    positives=(TARGET, "sum"),
    positive_rate=(TARGET, "mean"),
).reset_index()

missing_by_site = development.groupby(SITE)[FEATURES].apply(
    lambda frame: frame.isna().mean()
).T

print("Data source:", str(LOCAL_DATA_DIR) if LOCAL_DATA_DIR else "UCI URL fallback")
print("Development shape:", development.shape)
display(site_summary.round(4))
display(pd.crosstab(development[SITE], development[TARGET], margins=True))
display((missing_by_site * 100).round(1))

site_summary.to_csv(OUTPUT_DIR / "development_site_summary.csv", index=False)
missing_by_site.to_csv(OUTPUT_DIR / "development_missing_by_site.csv")

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)


def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors="coerce")

    # P1: sentinel đã được đọc thành NaN; thêm rule có căn cứ cho giá trị không hợp lệ.
    for column in ["trestbps", "chol"]:
        out.loc[out[column] <= 0, column] = np.nan

    return out


def make_preprocessor(scale_numeric=True):
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer([
        ("numerical", Pipeline(numeric_steps), NUMERICAL_FEATURES),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(
                strategy="most_frequent",
                add_indicator=True,
            )),
            ("encoder", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            )),
        ]), CATEGORICAL_FEATURES),
    ])


def make_preprocessor_compat(scale_numeric=True):
    try:
        return make_preprocessor(scale_numeric)
    except TypeError:
        numeric_steps = [
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ]
        if scale_numeric:
            numeric_steps.append(("scaler", StandardScaler()))
        return ColumnTransformer([
            ("numerical", Pipeline(numeric_steps), NUMERICAL_FEATURES),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent",
                    add_indicator=True,
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse=False,
                )),
            ]), CATEGORICAL_FEATURES),
        ])


def apply_smotenc(train_frame, seed):
    ready = apply_p1(train_frame).reset_index(drop=True)
    y = ready[TARGET].to_numpy(dtype="int8")

    counts = np.bincount(y, minlength=2)
    if counts.min() < 2:
        return ready

    # Impute từng cột thủ công để giữ đủ 13 cột. SimpleImputer mặc định
    # có thể loại cột toàn-missing trong một training fold, làm lệch
    # categorical_indices của SMOTENC.
    def fill_train_only(columns, strategy):
        filled_columns = []
        for column in columns:
            values = pd.to_numeric(ready[column], errors="coerce")
            if strategy == "median":
                fill_value = values.median()
            else:
                modes = values.dropna().mode()
                fill_value = modes.iloc[0] if not modes.empty else 0.0
            if pd.isna(fill_value):
                fill_value = 0.0
            filled_columns.append(
                values.fillna(float(fill_value)).to_numpy(dtype="float64")
            )
        return np.column_stack(filled_columns)

    x_numeric = fill_train_only(NUMERICAL_FEATURES, "median")
    x_categorical = fill_train_only(CATEGORICAL_FEATURES, "mode")
    x_for_smote = np.column_stack([x_numeric, x_categorical])

    categorical_indices = list(range(
        len(NUMERICAL_FEATURES),
        len(NUMERICAL_FEATURES) + len(CATEGORICAL_FEATURES),
    ))
    k_neighbors = max(1, min(5, int(counts.min()) - 1))

    sampler = SMOTENC(
        categorical_features=categorical_indices,
        k_neighbors=k_neighbors,
        random_state=seed,
    )
    x_resampled, y_resampled = sampler.fit_resample(x_for_smote, y)

    resampled = pd.DataFrame(
        x_resampled,
        columns=NUMERICAL_FEATURES + CATEGORICAL_FEATURES,
    )
    for column in CATEGORICAL_FEATURES:
        resampled[column] = np.rint(resampled[column]).astype(float)

    resampled[TARGET] = y_resampled.astype("int8")
    resampled[SITE] = "train_only_smotenc"
    return resampled[FEATURES + [TARGET, SITE]]


def frame_xy(frame):
    ready = apply_p1(frame)
    return ready[FEATURES], ready[TARGET].to_numpy(dtype="int8")

In [ ]:
def suggest_lr_params(trial):
    return {
        "C": trial.suggest_float("C", 0.01, 10.0, log=True),
        "solver": trial.suggest_categorical(
            "solver",
            ["lbfgs", "liblinear"],
        ),
        "positive_weight": trial.suggest_float(
            "positive_weight",
            0.75,
            2.0,
        ),
        "max_iter": 3000,
    }


def fit_lr_smotenc(train_frame, params, seed):
    seed_everything(seed)
    sampled = apply_smotenc(train_frame, seed)
    x_train, y_train = frame_xy(sampled)

    preprocessor = make_preprocessor_compat(scale_numeric=True)
    x_train = preprocessor.fit_transform(x_train).astype("float32")

    estimator = LogisticRegression(
        C=float(params["C"]),
        solver=str(params["solver"]),
        max_iter=int(params.get("max_iter", 3000)),
        class_weight={
            0: 1.0,
            1: float(params["positive_weight"]),
        },
        random_state=seed,
    )
    estimator.fit(x_train, y_train)
    return preprocessor, estimator


def transform_features(preprocessor, frame):
    x_frame, _ = frame_xy(frame)
    return preprocessor.transform(x_frame).astype("float32")


def predict_probability(preprocessor, estimator, frame, intercept=None):
    x_frame = transform_features(preprocessor, frame)
    raw_intercept = (
        float(estimator.intercept_[0])
        if intercept is None else float(intercept)
    )
    logits = x_frame @ estimator.coef_[0] + raw_intercept
    return expit(logits)


def tune_source_model(source_frame, seed, n_trials=N_TRIALS):
    labels = source_frame[TARGET].to_numpy(dtype="int8")
    groups = source_frame[SITE].to_numpy()
    splits = list(GroupKFold(n_splits=2).split(
        source_frame,
        labels,
        groups,
    ))

    def objective(trial):
        params = suggest_lr_params(trial)
        fold_auc = []

        for fold_number, (fit_idx, valid_idx) in enumerate(splits):
            fit_frame = source_frame.iloc[fit_idx].reset_index(drop=True)
            valid_frame = source_frame.iloc[valid_idx].reset_index(drop=True)

            preprocessor, estimator = fit_lr_smotenc(
                fit_frame,
                params,
                seed + fold_number,
            )
            probability = predict_probability(
                preprocessor,
                estimator,
                valid_frame,
            )
            fold_auc.append(roc_auc_score(
                valid_frame[TARGET],
                probability,
            ))

        return float(np.mean(fold_auc))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3),
    )
    started = time.perf_counter()
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=1,
        show_progress_bar=False,
    )

    best_params = dict(study.best_trial.params)
    best_params["max_iter"] = 3000
    return study, best_params, time.perf_counter() - started

In [ ]:
def sample_support_query(target_frame, support_per_class, seed):
    target_frame = target_frame.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    support_indices = []

    for label in [0, 1]:
        candidates = np.flatnonzero(
            target_frame[TARGET].to_numpy(dtype="int8") == label
        )
        if len(candidates) <= support_per_class:
            raise ValueError(
                f"Target site lacks query examples for class {label}: "
                f"n={len(candidates)}, support={support_per_class}"
            )
        chosen = rng.choice(
            candidates,
            size=support_per_class,
            replace=False,
        )
        support_indices.extend(chosen.tolist())

    support_indices = np.array(sorted(support_indices))
    all_indices = np.arange(len(target_frame))
    query_indices = np.setdiff1d(all_indices, support_indices)

    support = target_frame.iloc[support_indices].reset_index(drop=True)
    query = target_frame.iloc[query_indices].reset_index(drop=True)
    return support, query


def adapt_intercept(
    preprocessor,
    estimator,
    support_frame,
    prior_strength=INTERCEPT_PRIOR_STRENGTH,
    max_iter=25,
):
    x_support = transform_features(preprocessor, support_frame)
    y_support = support_frame[TARGET].to_numpy(dtype="float64")
    coef = estimator.coef_[0]
    base_intercept = float(estimator.intercept_[0])
    intercept = base_intercept

    # Newton update cho intercept; coefficients được khóa hoàn toàn.
    for _ in range(max_iter):
        probability = expit(x_support @ coef + intercept)
        gradient = (
            np.sum(probability - y_support)
            + prior_strength * (intercept - base_intercept)
        )
        hessian = (
            np.sum(probability * (1.0 - probability))
            + prior_strength
        )
        step = gradient / max(hessian, 1e-8)
        intercept -= step
        if abs(step) < 1e-7:
            break

    return float(intercept)


def score_probability(y_true, probability):
    prediction = (probability >= THRESHOLD).astype("int8")
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    return {
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "pr_auc": average_precision_score(y_true, probability),
        "specificity": (
            tn / (tn + fp) if (tn + fp) else np.nan
        ),
        "f1": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "roc_auc": (
            roc_auc_score(y_true, probability)
            if len(np.unique(y_true)) > 1 else np.nan
        ),
        "brier": brier_score_loss(y_true, probability),
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "n_query": int(len(y_true)),
    }

## Chạy thí nghiệm Few-shot

Mỗi dòng kết quả dùng cùng một query set cho hai nhánh:

- no_adaptation: LR + SMOTENC từ 2 source hospitals, không dùng support để sửa model.
- fewshot_intercept: dùng support labels để cập nhật intercept; coefficients và threshold vẫn giữ nguyên.

Không chọn support size hoặc seed dựa trên query metrics trong notebook này. Việc kết luận phải dựa trên bảng tổng hợp development và được khóa trước khi chạy external holdout.

In [ ]:
result_rows = []
tuning_rows = []

for target_site in DEVELOPMENT_SITES:
    source_sites = [
        site for site in DEVELOPMENT_SITES
        if site != target_site
    ]
    source_frame = development[
        development[SITE].isin(source_sites)
    ].reset_index(drop=True)
    target_frame = development[
        development[SITE] == target_site
    ].reset_index(drop=True)

    for model_seed in MODEL_SEEDS:
        study, best_params, tuning_seconds = tune_source_model(
            source_frame,
            seed=model_seed,
            n_trials=N_TRIALS,
        )
        preprocessor, estimator = fit_lr_smotenc(
            source_frame,
            best_params,
            seed=model_seed,
        )

        tuning_rows.append({
            "target_site": target_site,
            "source_sites": "+".join(source_sites),
            "model_seed": model_seed,
            "inner_cv_roc_auc": float(study.best_value),
            "n_trials": len(study.trials),
            "tuning_seconds": tuning_seconds,
            "best_params": json.dumps(
                best_params,
                sort_keys=True,
            ),
        })

        for support_per_class in SUPPORT_PER_CLASS:
            for support_seed in SUPPORT_SEEDS:
                support, query = sample_support_query(
                    target_frame,
                    support_per_class=support_per_class,
                    seed=support_seed,
                )

                base_probability = predict_probability(
                    preprocessor,
                    estimator,
                    query,
                )
                adapted_intercept = adapt_intercept(
                    preprocessor,
                    estimator,
                    support,
                )
                adapted_probability = predict_probability(
                    preprocessor,
                    estimator,
                    query,
                    intercept=adapted_intercept,
                )

                for adaptation, probability in [
                    ("no_adaptation", base_probability),
                    ("fewshot_intercept", adapted_probability),
                ]:
                    result_rows.append({
                        "evaluation": "development_fewshot_simulation",
                        "target_site": target_site,
                        "source_sites": "+".join(source_sites),
                        "model": "Logistic Regression",
                        "sampling": "SMOTENC_train_only",
                        "adaptation": adaptation,
                        "support_per_class": support_per_class,
                        "support_total": int(len(support)),
                        "support_seed": support_seed,
                        "model_seed": model_seed,
                        "intercept_prior_strength": INTERCEPT_PRIOR_STRENGTH,
                        "adapted_intercept": (
                            adapted_intercept
                            if adaptation == "fewshot_intercept"
                            else float(estimator.intercept_[0])
                        ),
                        **score_probability(
                            query[TARGET].to_numpy(),
                            probability,
                        ),
                    })

results = pd.DataFrame(result_rows)
tuning_results = pd.DataFrame(tuning_rows)

print("Raw result rows:", len(results))
display(tuning_results.round(5).head(12))
display(results.round(5).head(12))

In [ ]:
def aggregate_metrics(frame, group_columns):
    return frame.groupby(group_columns).agg(
        accuracy_mean=("accuracy", "mean"),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
        specificity_mean=("specificity", "mean"),
        f1_mean=("f1", "mean"),
        roc_auc_mean=("roc_auc", "mean"),
        brier_mean=("brier", "mean"),
        false_negatives_mean=("false_negatives", "mean"),
        false_positives_mean=("false_positives", "mean"),
        n_runs=("accuracy", "size"),
    ).reset_index()


site_summary = aggregate_metrics(
    results,
    ["target_site", "adaptation", "support_per_class"],
)

overall_summary = aggregate_metrics(
    results,
    ["adaptation", "support_per_class"],
)

worst_site_summary = site_summary.groupby(
    ["adaptation", "support_per_class"]
).agg(
    accuracy_worst=("accuracy_mean", "min"),
    recall_worst=("recall_mean", "min"),
    pr_auc_worst=("pr_auc_mean", "min"),
    roc_auc_worst=("roc_auc_mean", "min"),
    specificity_worst=("specificity_mean", "min"),
    brier_worst=("brier_mean", "max"),
).reset_index()

paired = results.pivot_table(
    index=[
        "target_site",
        "support_per_class",
        "support_seed",
        "model_seed",
    ],
    columns="adaptation",
    values=[
        "accuracy",
        "recall",
        "pr_auc",
        "roc_auc",
        "specificity",
        "f1",
        "brier",
    ],
)
paired.columns = [
    f"{metric}_{adaptation}"
    for metric, adaptation in paired.columns
]
paired = paired.reset_index()

for metric in [
    "accuracy",
    "recall",
    "pr_auc",
    "roc_auc",
    "specificity",
    "f1",
    "brier",
]:
    paired[f"delta_{metric}"] = (
        paired[f"{metric}_fewshot_intercept"]
        - paired[f"{metric}_no_adaptation"]
    )

paired_delta_summary = paired.groupby(
    ["target_site", "support_per_class"]
).agg(
    delta_accuracy_mean=("delta_accuracy", "mean"),
    delta_recall_mean=("delta_recall", "mean"),
    delta_pr_auc_mean=("delta_pr_auc", "mean"),
    delta_roc_auc_mean=("delta_roc_auc", "mean"),
    delta_specificity_mean=("delta_specificity", "mean"),
    delta_f1_mean=("delta_f1", "mean"),
    delta_brier_mean=("delta_brier", "mean"),
).reset_index()

print("Per-target development summary")
display(site_summary.round(5))

print("Overall development summary")
display(overall_summary.round(5))

print("Worst-target-site summary")
display(worst_site_summary.round(5))

print("Paired few-shot minus no-adaptation deltas")
display(paired_delta_summary.round(5))

In [ ]:
plot_frame = overall_summary.melt(
    id_vars=["adaptation", "support_per_class"],
    value_vars=[
        "accuracy_mean",
        "recall_mean",
        "roc_auc_mean",
    ],
    var_name="metric",
    value_name="value",
)
plot_frame["support_label"] = (
    plot_frame["support_per_class"].astype(str)
    + " per class"
)

sns.set_theme(style="whitegrid")
g = sns.catplot(
    data=plot_frame,
    x="metric",
    y="value",
    hue="adaptation",
    col="support_label",
    kind="bar",
    height=4,
    aspect=1.15,
)
g.set_axis_labels("", "Development mean")
g.set_titles("Support: {col_name}")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "fewshot_development_metrics.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

worst_plot = worst_site_summary.melt(
    id_vars=["adaptation", "support_per_class"],
    value_vars=[
        "accuracy_worst",
        "recall_worst",
        "roc_auc_worst",
    ],
    var_name="metric",
    value_name="value",
)
sns.catplot(
    data=worst_plot,
    x="metric",
    y="value",
    hue="adaptation",
    col="support_per_class",
    kind="bar",
    height=4,
    aspect=1.15,
)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
results.to_csv(
    OUTPUT_DIR / "fewshot_raw_results.csv",
    index=False,
)
tuning_results.to_csv(
    OUTPUT_DIR / "fewshot_tuning_results.csv",
    index=False,
)
site_summary.to_csv(
    OUTPUT_DIR / "fewshot_site_summary.csv",
    index=False,
)
overall_summary.to_csv(
    OUTPUT_DIR / "fewshot_overall_summary.csv",
    index=False,
)
worst_site_summary.to_csv(
    OUTPUT_DIR / "fewshot_worst_site_summary.csv",
    index=False,
)
paired.to_csv(
    OUTPUT_DIR / "fewshot_paired_results.csv",
    index=False,
)
paired_delta_summary.to_csv(
    OUTPUT_DIR / "fewshot_paired_delta_summary.csv",
    index=False,
)

run_config = {
    "notebook": "28_UCI_Multicenter_FewShot_LR_SMOTENC_Colab",
    "protocol": {
        "development_sites": DEVELOPMENT_SITES,
        "target_site_rotation": (
            "one target development hospital; two source development hospitals"
        ),
        "inner_cv": "GroupKFold(2) over source hospitals",
        "external_holdout": "Hungarian not loaded or evaluated",
    },
    "model": "Logistic Regression",
    "sampling": (
        "SMOTENC fit only inside each source training split; "
        "categorical columns are never one-hot encoded before SMOTENC"
    ),
    "preprocessing": (
        "P1, train-only median/mode imputation, missing indicators, "
        "one-hot encoding, numeric scaling"
    ),
    "fewshot": {
        "support_per_class": list(SUPPORT_PER_CLASS),
        "support_seeds": list(SUPPORT_SEEDS),
        "adaptation": (
            "intercept-only Newton update; coefficients frozen; "
            "Gaussian prior keeps update conservative"
        ),
        "intercept_prior_strength": INTERCEPT_PRIOR_STRENGTH,
        "threshold": THRESHOLD,
    },
    "optuna": {
        "enabled": True,
        "n_trials": N_TRIALS,
        "search_metric": "mean source-hospital validation ROC-AUC",
    },
    "model_seeds": list(MODEL_SEEDS),
    "test_policy": (
        "query labels are used only for reporting metrics; "
        "no support size, seed, threshold or model decision is chosen from query"
    ),
    "interpretation": (
        "development simulation only; an external blind claim requires an "
        "untouched holdout and no post-hoc tuning"
    ),
}
(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8",
)

print("Saved artifacts to:", OUTPUT_DIR)
try:
    import shutil
    zip_path = shutil.make_archive(
        "/content/uci_multicenter_fewshot_lr_smotenc_results",
        "zip",
        OUTPUT_DIR,
    )
    print("ZIP:", zip_path)
except Exception as exc:
    print("ZIP creation skipped:", exc)

## Checklist Colab 28

- [x] LR + SMOTENC là model chính.
- [x] Optuna chỉ tune trên 2 source development hospitals.
- [x] SMOTENC chỉ fit trong training fold/source train.
- [x] P1, imputation, missing indicators, one-hot và scaling đều train-only.
- [x] Xoay target-site qua Cleveland, Switzerland và VA.
- [x] Support set nhỏ, phân tầng theo class.
- [x] Few-shot chỉ cập nhật intercept; coefficients bị khóa.
- [x] Query set tách khỏi support và chỉ dùng để báo cáo metric.
- [x] So sánh paired no_adaptation với fewshot_intercept.
- [x] Báo cáo mean và worst target-site cho Accuracy, Recall, ROC-AUC, PR-AUC, Specificity, F1 và Brier.
- [x] Không load hoặc đánh giá Hungarian trong Colab 28.
- [ ] Không chọn lại protocol sau khi xem Hungarian.
- [ ] Nếu cần external evaluation mù tuyệt đối sau các thí nghiệm đã xem, bổ sung hospital/dataset thứ 5.